*Работа выполнялась Зотовой Эрикой (группа БКЛ241) и Есиным Александром (мобильность с ФПЛ)*

##Основная работа

###Необходимые импорты

Анализ данных и визуализация

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

NLTK и регулярки для дальнейшей предобработки текстов отзывов

In [2]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

Инструменты машинного обучения (включая метрики, векторизатор и инструменты масштабирования)

In [3]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix #это в дальнейшем использовалось для масштабирования признаков

###Предобработка данных

In [4]:
test_df = pd.read_csv("test.csv")
train_df = pd.read_csv("train.csv")

In [5]:
train_df
#изначально таблица выглядит так (ну, вы знаете)

,id,text,answer
0,0,What a disappointment... admittedly the best o...,0
1,1,This is a pale imitation of the Die Hard franc...,0
2,2,"This good-guy-vs-the-evil-tyrant story, set in...",0
3,3,This is a documentary I came across by chance ...,1
4,4,This installment of Masters of Horror was terr...,0
...,...,...,...
24995,24995,"Horrible Script, which was apparently directed...",0
24996,24996,Five years on from the Tenko survivors returni...,1
24997,24997,"I don't understand. Not being a critic, i am n...",1
24998,24998,"This movie was pretentious, foppish and just d...",0


Функция предобработки текста: лемматизация, приведение к нижнему регистру

*Все из этого очень ненамного, но улучшило качество модели – поэтому мы оставили эту функцию, она не занимает много памяти и времени и какое-то значение все же имеет.*

In [6]:
nltk.download('wordnet')

def preprocessing(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
df_train_cossim = train_df.copy()
#это таблица для дальнейшей работы, в т.ч. с функциями косинусной близости

In [8]:
df_train_cossim['normalized_text'] = df_train_cossim['text'].apply(preprocessing)

In [9]:
df_train_cossim
#а так выглядит таблица со столбцом, содержащим обработанный текст

,id,text,answer,normalized_text
0,0,What a disappointment... admittedly the best o...,0,what a disappointment admittedly the best of t...
1,1,This is a pale imitation of the Die Hard franc...,0,this is a pale imitation of the die hard franc...
2,2,"This good-guy-vs-the-evil-tyrant story, set in...",0,this goodguyvstheeviltyrant story set in 19th ...
3,3,This is a documentary I came across by chance ...,1,this is a documentary i came across by chance ...
4,4,This installment of Masters of Horror was terr...,0,this installment of master of horror wa terrib...
...,...,...,...,...
24995,24995,"Horrible Script, which was apparently directed...",0,horrible script which wa apparently directed b...
24996,24996,Five years on from the Tenko survivors returni...,1,five year on from the tenko survivor returning...
24997,24997,"I don't understand. Not being a critic, i am n...",1,i dont understand not being a critic i am not ...
24998,24998,"This movie was pretentious, foppish and just d...",0,this movie wa pretentious foppish and just dow...


###Функции определения интонации

Мы не использовали готовые библиотеки для анализа интонации, но сделали две функции определения косинусной близости текста отзыва и некоторого массива популярных слов с позитивной и негативной интонацией.

*Списки слов составлялись вручную – одним из потенциальных увеличений всего "проекта" могло бы быть более осмысленное написание этих функций, но сейчас это было не очень рационально, так как итоговое влияние на качество модели логистической регрессии оказалось не таким большим. В последнем разделе мы скажем о том, почему все-таки использовали этот инструмент.*

In [10]:
def positive_checker(text: str):

    positive_words = [
        'good', 'great', 'excellent', 'amazing', 'wonderful',
        'fantastic', 'perfect', 'love', 'beautiful', 'awesome',
        'brilliant', 'outstanding', 'superb', 'terrific', 'magnificent',
        'happy', 'glad', 'pleased', 'delighted', 'enjoy',
        'nice', 'lovely', 'best', 'favorite', 'positive',
        'entertaining', 'engaging', 'impressive', 'beautiful', 'stunning',
        'masterpiece', 'brilliant', 'flawless', 'incredible', 'awesome',
        'recommend', 'must-see', 'worth', 'best', 'favorite',
        'well-acted', 'well-written', 'well-directed', 'thought-provoking',
        'hilarious', 'funny', 'emotional', 'powerful', 'moving'
        ]
    processed_text = text
    processed_reference = ' '.join(positive_words)

    vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)

    #была мысль также использовать модели word2vec-формата, но
    #они работают дольше, весят больше, а эффективность в данном контексте ниже

    tfidf_matrix = vectorizer.fit_transform([processed_text, processed_reference])

    similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])

    return similarity[0][0]

In [11]:
def negative_checker(text: str):
    negative_words = [
        'bad', 'terrible', 'awful', 'horrible', 'poor',
        'disgusting', 'hate', 'worst', 'dreadful', 'atrocious',
        'boring', 'stupid', 'useless', 'waste', 'mediocre',
        'disappointing', 'frustrating', 'annoying', 'irritating',
        'wrong', 'fake', 'broken', 'failure', 'upset', 'sad',
        'not good', 'waste of time', 'waste of money',
        'fell asleep', 'turned off', 'could not finish',
        'do not recommend', 'would not recommend',
        'avoid this', 'stay away', 'save your money'
    ]
    processed_text = text
    processed_reference = ' '.join(negative_words)

    vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)
    tfidf_matrix = vectorizer.fit_transform([processed_text, processed_reference])

    similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
    return similarity[0][0]

###Получение значений косинусной близости и векторов

*Этот код занимает много времени и памяти – над скоростью можно было бы поработать, но здесь приоритизировалось качество модели, а скорость на него не влияет*

In [12]:
df_train_cossim['neg_sim'] = df_train_cossim['normalized_text'].apply(negative_checker)
df_train_cossim['pos_sim'] = df_train_cossim['normalized_text'].apply(positive_checker)

In [13]:
df_train_cossim
#а так выглядит таблица с новыми данными

,id,text,answer,normalized_text,neg_sim,pos_sim
0,0,What a disappointment... admittedly the best o...,0,what a disappointment admittedly the best of t...,0.022440,0.004819
1,1,This is a pale imitation of the Die Hard franc...,0,this is a pale imitation of the die hard franc...,0.028418,0.001322
2,2,"This good-guy-vs-the-evil-tyrant story, set in...",0,this goodguyvstheeviltyrant story set in 19th ...,0.012739,0.000000
3,3,This is a documentary I came across by chance ...,1,this is a documentary i came across by chance ...,0.049160,0.012765
4,4,This installment of Masters of Horror was terr...,0,this installment of master of horror wa terrib...,0.020298,0.002720
...,...,...,...,...,...,...
24995,24995,"Horrible Script, which was apparently directed...",0,horrible script which wa apparently directed b...,0.010153,0.001710
24996,24996,Five years on from the Tenko survivors returni...,1,five year on from the tenko survivor returning...,0.022483,0.007914
24997,24997,"I don't understand. Not being a critic, i am n...",1,i dont understand not being a critic i am not ...,0.037552,0.001595
24998,24998,"This movie was pretentious, foppish and just d...",0,this movie wa pretentious foppish and just dow...,0.045760,0.007346


*Так мы получали вектора tf-idf; мы добавили биграммы и увеличили количество признаков настолько, насколько позволяла среда выполнения (colab не позволял больше 20000 – сеанс прерывался, – vscode выдерживает еще меньше). По нашим наблюдениям это улучшило качество модели примерно на 0,01 (так же, как и вышеприведенные признаки).*

In [14]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=20000)
train_vec = vectorizer.fit_transform(df_train_cossim["text"])
#x_train = train_vec.toarray()

#

In [15]:
y_train = df_train_cossim["answer"].to_numpy()

Теперь то же самое делаем с тестовой выборкой: модель нельзя обучить и проверять на разном количестве (*и качестве, желательно*) признаков, поэтому, если мы брали для обучающей и валидационной выборок признаки, связанные с интонационной проверкой, и предобрабатывали текст – то же самое нужно сделать и с тестовой.

In [16]:
test_df['text_normalized'] = test_df['text'].apply(preprocessing)

In [17]:
test_vec = vectorizer.transform(test_df["text"])
x_test = test_vec.toarray()

Здесь (при векторизации самого текста) был использован первичный, а не обработанный текст – экспериментально было проверено, что, если использовать нормализованный текст именно для обучения, качество падало до 0.85; мы не поняли, с чем это можно связать.

In [18]:
test_df['neg_sim'] = test_df['text_normalized'].apply(negative_checker)
test_df['pos_sim'] = test_df['text_normalized'].apply(positive_checker)

###Подготовка признаков и обучение модели

Это – один из способов масштабирования; без него модель, видимо, не считывала признаки с косинусной близостью как достаточно значимые и из-за этого точность не менялась практически совсем. Необходимо, конечно, также применить его и к тестовой, и к обучающей выборке.

In [19]:
pos_sim_sparse = csr_matrix(df_train_cossim["pos_sim"].values.reshape(-1, 1))
neg_sim_sparse = csr_matrix(df_train_cossim["neg_sim"].values.reshape(-1, 1))

x_train_sparse = hstack([train_vec, pos_sim_sparse, neg_sim_sparse])

In [20]:
test_pos_sim_sparse = csr_matrix(test_df["pos_sim"].values.reshape(-1, 1))
test_neg_sim_sparse = csr_matrix(test_df["neg_sim"].values.reshape(-1, 1))

x_test = hstack([test_vec, test_pos_sim_sparse, test_neg_sim_sparse])

Начинаем обучение модели; размер тестовой выборки был выведен экспериментально

In [21]:
x_train_sparse, x_val, y_train, y_val = train_test_split(
    x_train_sparse, y_train, test_size=0.24, random_state=42
)

Здесь происходит подбор параметров – мы использовали тот же принцип, что и с knn, но с весами лог. регрессии.

In [22]:
best_c = -1
best_value = 0
for c in [0.1, 0.5, 1.0, 2.0, 5.0, 7.0, 10.0]:
    logreg = LogisticRegression(random_state=42, C=c, max_iter=10000)
    logreg.fit(x_train_sparse, y_train)
    pred = logreg.predict(x_val)
    print(f"C={c}: {f1_score(y_val, pred, average='weighted'):.4f}")

    if f1_score(y_val, pred, average='weighted') > best_value:
        best_value = f1_score(y_val, pred, average='weighted')
        best_c = c

C=0.1: 0.8578
C=0.5: 0.8828
C=1.0: 0.8910
C=2.0: 0.8945
C=5.0: 0.8965
C=7.0: 0.8963
C=10.0: 0.8955


In [23]:
logreg = LogisticRegression(random_state=42, C=best_c, max_iter=10000)
logreg.fit(x_train_sparse, y_train)

LogisticRegression(C=5.0, max_iter=10000, random_state=42)

In [24]:
pred = logreg.predict(x_val)
print(classification_report(y_val, pred))

              precision    recall  f1-score   support

           0       0.91      0.89      0.90      3038
           1       0.89      0.91      0.90      2962

    accuracy                           0.90      6000
   macro avg       0.90      0.90      0.90      6000
weighted avg       0.90      0.90      0.90      6000



###Выведение результата

In [28]:
test_predictions = logreg.predict(x_test)
result = test_df[["id"]].copy()
result["answer"] = test_predictions
result.to_csv("result.csv", index=False)

##О ходе работы и других попытках улучшить качество

###Модель с качеством 0,87

Сначала мы попробовали сделать просто самую простую модель (на 6); вот как это выглядело:

In [34]:
test_df = pd.read_csv("test.csv")
train_df = pd.read_csv("train.csv")

vectorizer = TfidfVectorizer(max_features=1000)
train_vec = vectorizer.fit_transform(train_df["text"])
#best_words_indices = np.asarray(train_vec.sum(axis=1)).ravel().argsort()[::-1][:1000]
x_train = train_vec.toarray()
x_train

y_train = train_df["answer"].to_numpy()

x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.33, random_state=42
    )


test_vec = vectorizer.transform(test_df["text"])
x_test = test_vec.toarray()
x_test

logreg = LogisticRegression(random_state=42)
logreg.fit(x_train, y_train)

pred = logreg.predict(x_val)
pred

print(classification_report(pred, y_val))

#она, кстати, честно показывала 0,87 – почему-то сейчас 0,86, но кажется с учетом итога уже no matter

              precision    recall  f1-score   support

           0       0.86      0.86      0.86      4118
           1       0.86      0.86      0.86      4132

    accuracy                           0.86      8250
   macro avg       0.86      0.86      0.86      8250
weighted avg       0.86      0.86      0.86      8250



Разные функции для KNN

Далее в попытках понять, как можно улучшить модель, мы решили немного упростить задачу и сначала посмотреть, как добавление или убирание разных признаков работали на knn.


**Что мы пытались делать:**


*   удалять стоп-слова (это не помогло больше чем на одну сотую, так же как и лемматизация, а впоследствии с логистической регрессией только ухудшило качество)
*   добавлять в качестве признака длину отзыва (это значительно ухудшило качество)
*   отделять от текста части речи (см. ниже функцию) – это никак не повлияло на результат





In [25]:
def adj_detector(text: list):
    adj_list = ['JJ', 'JJR', 'JJS']
    token_list = []
    for tag in nltk.pos_tag(text):
        if tag[1] in adj_list:
            token_list.append(tag[0])
    return token_list

def verb_detector(text: list):
    verb_list = ['VB', 'VBD', 'VBG', 'VBN']
    token_list = []
    for tag in nltk.pos_tag(text):
        if tag[1] in verb_list:
            token_list.append(tag[0])
    return token_list



*   считать общее среднее значение векторов по каждому отзыву (до этой функции был импорт word2vec-модели)



In [ ]:
def vector_sum(text: list):
    mean_vectors = {}
    all_vectors = []
    for word in text:
        if word in model_word2vec:
              vectors = model_word2vec[word][:120]
              all_vectors.append(vectors)

        else:
              pass

    vectors_array = np.array(all_vectors)
    mean_vector = np.mean(vectors_array, axis=0)
    mean_value = np.mean(mean_vector)
    return mean_value

Но качественно увеличило качество предсказаний модели только добавление векторных значений для позитивной/негативной интонации – такого разрыва в значениях не было на логистической регрессии, но то значение, которое это изменение имело для простой классификации, показалось нам достаточным для того, чтобы добавить ее и в регрессию тоже.